# Train Apprentice's SQL judge v2 (fine-tune from v1)

v1 scores 99.3% on held-out questions from the 16 templates it trained on, but 76.8% on unseen question types, and it approved a wrong answer to a Hinglish question.
v2 starts from v1 and adds 4 of the 8 unseen templates, real student attempts and Hinglish rewrites (`bench/judge_data_v2.py` in [Ayush-e4/apprentice](https://github.com/Ayush-e4/apprentice)).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Safe to run in the session that already serves v1 on port 8009: Kev is reused if installed, everything here goes under `/content/judge-v2`, and port 8009 is never touched.

Steps: install Kev → download data → download v1 → fine-tune → score v1 and v2 → save → (optional) serve v2 on port 8010.

In [ ]:
REPO = 'Ayush-e4/apprentice'
BRANCH = 'main'
BASE = 'Qwen/Qwen3.5-0.8B-Base'
BASE_REVISION = 'dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68'  # the base Kev-0.8B was trained on
V1 = '/content/v1/sql-judge'
WORK = '/content/judge-v2'  # separate from v1's files
OUT = 'runs/sql-judge-v2'
import os
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# Install Kev from source (it is not on PyPI; `pip install kev` is a different package). Skipped if this session has it.
if not os.path.exists('/content/kev'):
    !git clone -q --depth 1 https://github.com/jaredpalmer/kev.git /content/kev
    !pip install -q -e '/content/kev[serve]'
# Kev pins torch<2.9, which downgrades Colab's torch. Colab's torchvision is built for the newer torch and
# breaks `import transformers` ("operator torchvision::nms does not exist"), and its torchao is older than
# peft accepts ("only versions above 0.16.0 are supported"). Kev uses neither.
!pip uninstall -y -q torchvision torchaudio torchao
!python -c "import kev, torch, transformers, peft; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# Download the labelled data: v2 train, and the three held-out files neither v1 nor v2 trains on
raw = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/data'
FILES = {'train': 'judge_v2/train.jsonl', 'seen': 'judge/heldout.jsonl',
         'unseen': 'judge_v2/heldout_unseen.jsonl', 'hinglish': 'judge_v2/heldout_hinglish.jsonl'}
!mkdir -p data
for name, path in FILES.items():
    !wget -q {raw}/{path} -O data/{name}.jsonl
    assert os.path.getsize(f'data/{name}.jsonl') > 0, f'download failed: {path}'
!wc -l data/*.jsonl

In [ ]:
# Download the v1 judge (the starting point, and the baseline)
if not os.path.exists(f'{V1}/head.pt'):
    !wget -q https://github.com/{REPO}/releases/download/judge-v1/sql-judge.zip -O /content/v1-sql-judge.zip
    !mkdir -p /content/v1 && unzip -q -o /content/v1-sql-judge.zip -d /content/v1
assert os.path.exists(f'{V1}/head.pt'), 'v1 download failed: read the error above'

In [ ]:
# Fine-tune from v1, small learning rate. fp32 because the T4 has no bf16 support.
# Batch 2 x accum 4 (same effective batch as v1's 4 x 2) leaves GPU memory for a v1 server already running.
!python -m kev.train --data data/train.jsonl --base {BASE} --base_revision {BASE_REVISION} \
    --init_from {V1} --epochs 2 --lr 2e-5 --batch 2 --accum 4 --dtype fp32 \
    --checkpointing 1 --device cuda --out {OUT}
assert os.path.exists(f'{OUT}/head.pt'), 'training failed: read the error above'

In [ ]:
# Score v1 and v2 on the same three held-out files
RUNS = {'v1': V1, 'v2': OUT}
for model, run in RUNS.items():
    for name in ('seen', 'unseen', 'hinglish'):
        out = f'runs/eval-{model}-{name}'
        !python -m kev.benchmark --run {run} --data data/{name}.jsonl --device cuda --out {out}
        assert os.path.exists(f'{out}/report.json'), f'{model} on {name} failed: read the error above'

In [ ]:
# Side by side: accuracy (higher is better) and ECE, the calibration error (lower is better)
import json
print(f"{'held-out':<10}{'n':>6}{'v1 acc':>9}{'v2 acc':>9}{'v1 ECE':>9}{'v2 ECE':>9}")
for name in ('seen', 'unseen', 'hinglish'):
    r = {m: json.load(open(f'runs/eval-{m}-{name}/report.json'))['clean'] for m in RUNS}
    print(f"{name:<10}{r['v2']['n']:>6}{r['v1']['acc']:>9.1%}{r['v2']['acc']:>9.1%}{r['v1']['ece']:>9.3f}{r['v2']['ece']:>9.3f}")

In [ ]:
# Save the v2 judge and its eval reports to your computer
!cd runs && zip -qr sql-judge-v2.zip sql-judge-v2 eval-v1-* eval-v2-*
assert os.path.exists('runs/sql-judge-v2.zip'), 'zip failed'
from google.colab import files
files.download('runs/sql-judge-v2.zip')

## Optional: serve v2 from Colab

Serves v2 on **port 8010** with its own Cloudflare quick tunnel and a new API key. A v1 server on port 8009 keeps running.
Put the printed values in Apprentice's `.env` as `JUDGE_BASE_URL` and `JUDGE_API_KEY`. The URL stops working when this Colab session ends.

In [ ]:
import re, secrets, subprocess, time
key_v2 = secrets.token_hex(24)
env = {**os.environ, 'KEV_API_KEY': key_v2}
server_v2 = subprocess.Popen(['python', '-m', 'kev.serve', '--run', OUT, '--port', '8010', '--device', 'cuda'],
                             env=env, stdout=open('serve-v2.log', 'w'), stderr=subprocess.STDOUT)
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel_v2 = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8010'],
                             stdout=open('tunnel-v2.log', 'w'), stderr=subprocess.STDOUT)
url_v2 = None
for _ in range(60):
    time.sleep(2)
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel-v2.log').read())
    if found:
        url_v2 = found.group(0)
        break
print('JUDGE_BASE_URL=' + str(url_v2))
print('JUDGE_API_KEY=' + key_v2)
print('JUDGE_MODEL=kev-sql-judge-v2')